# 03 · Escenarios — IVR Alkosto

Toma `df_transformado` (02_limpieza_segmentacion.ipynb) y hace lo que hacía la segunda mitad de
`01_Alk_final.ipynb`: identifica el escenario/desenlace final de cada
`id_conversacion` (el viejo `traza_final`), filtra por escenario y por cantidad
de pasos, y exporta los `base_*.xlsx` que consume el Notebook 5 (04_grafos.ipynb).

#### Corrección importante: las aristas del grafo se recalculan sobre la secuencia filtrada a solo pasos de negocio, no sobre la secuencia completa

`op_text_final` en `df_transformado` se calculó (02_limpieza_segmentacion.ipynb) sobre la secuencia
**completa** de cada conversación — a propósito, para no perder el orden real. Pero
eso significa que, para una fila de negocio, `op_text_final` frecuentemente apunta
a un paso **técnico** (ej. `Menu principal → Consulta ws ConsultaHabeasData`), no al
siguiente paso de negocio. Si exportáramos esa columna tal cual al grafo, cada nodo
de negocio terminaría conectado mayormente a nodos técnicos en vez de a la
siguiente opción real elegida.

Por eso acá:
1. Filtramos a `incluir_en_grafo == True` (solo navegación de negocio + paso a
   asesor, ver 02_limpieza_segmentacion.ipynb).
2. **Recalculamos `op_text_final` sobre esa secuencia ya reducida** — así una fila
   de negocio queda conectada directamente a la siguiente fila de negocio,
   saltándose los pasos técnicos intermedios sin corromper el orden.

#### Corrección de umbrales: contar solo pasos de negocio

Los umbrales originales ("hasta 4/5/6 opciones") se calibraron sobre datos donde
*todos* los pasos eran de navegación. Con los datos actuales, una conversación
tiene en promedio ~13 pasos totales pero muchos son técnicos — si contáramos pasos
totales, casi todo caería en "más de N". Por eso `n_pasos` se cuenta **solo sobre
los pasos de negocio** (`incluir_en_grafo == True`), que es la magnitud comparable
a lo que medía el pipeline original.

#### Cruce con `Ultima traza` (hoja del maestro)

Esta hoja (110 filas: `NombreUltimaTraza`, `Pasa a Agente`, `Efectivo Ultima
Traza`, `ClasificaciónEfectivos`) es el reemplazo directo del viejo `traza_final`
+ `base_final.xlsx`. Trae, además, una clasificación de "efectivo" por bucket
(`SI Despacho`, `SI Garantias`, `Si Horarios`) que el pipeline original no tenía —
se conserva como columna de auditoría en el resumen, aunque los 5 escenarios que
exportamos abajo siguen siendo los mismos nombrados en `01_Alk_final.ipynb` para
no romper continuidad con los grafos ya existentes.

**Entrada:** `df_transformado_<periodo>.parquet` (Notebook 3) +
`Maestro_trazas_alkosto.xlsx` (hoja `Ultima traza`).

**Salida:** `base_<escenario>_hasta_N.xlsx` / `_mas_N.xlsx` por escenario, listos
para el Notebook 5.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from unidecode import unidecode

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## Parámetros — deben coincidir con los notebooks anteriores

In [ ]:
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
PROC_DIR = DATA_DIR / "02_procesado"
ESCEN_DIR = DATA_DIR / "03_escenarios"
ESCEN_DIR.mkdir(parents=True, exist_ok=True)

MAESTRO_PATH = Path("C:/Users/jupasoro/OneDrive - Emtelco/Proyectos/Alkosto/grafos_ivr_2026/Maestro trazas alkosto.xlsx")
IN_TRANSFORMADO_PATH = PROC_DIR / f"df_transformado_{FECHA_INICIO}_{FECHA_FIN}.parquet"

assert IN_TRANSFORMADO_PATH.exists(), f"No encuentro {IN_TRANSFORMADO_PATH} — corre primero 02_limpieza_segmentacion.ipynb"
assert MAESTRO_PATH.exists(), f"No encuentro {MAESTRO_PATH} — ajusta MAESTRO_PATH"
print(f"Leyendo: {IN_TRANSFORMADO_PATH}")

Leyendo: data\02_procesado\df_transformado_2026-06-01_2026-06-30.parquet


In [ ]:
df_transformado = pd.read_parquet(IN_TRANSFORMADO_PATH)
print("df_transformado:", df_transformado.shape)
print("incluir_en_grafo:", df_transformado["incluir_en_grafo"].value_counts().to_dict())

df_transformado: (1296067, 27)
incluir_en_grafo: {False: 792460, True: 503607}


## Reducir a pasos de negocio y recalcular las aristas

`df_negocio` es la secuencia que realmente vamos a graficar: solo pasos con
`incluir_en_grafo == True`, reordenados, con `op_text_final` recalculado dentro
de esa secuencia reducida (no la del Notebook 3, que era sobre la traza completa).

In [ ]:
df_negocio = (
    df_transformado[df_transformado["incluir_en_grafo"]]
    .sort_values(["id_conversacion", "orden"])
    .reset_index(drop=True)
    .copy()
)

# Recalculamos la arista sobre la secuencia YA reducida a pasos de negocio.
df_negocio["op_text_final"] = df_negocio.groupby("id_conversacion")["op_text"].shift(-1)
ultimas_filas_negocio = df_negocio.groupby("id_conversacion").tail(1).index
df_negocio.loc[ultimas_filas_negocio, "op_text_final"] = "final"

df_negocio["orden_negocio"] = df_negocio.groupby("id_conversacion").cumcount()

print("df_negocio (solo pasos de negocio, aristas recalculadas):", df_negocio.shape)
print("id_conversacion con al menos un paso de negocio:", df_negocio["id_conversacion"].nunique())
df_negocio[["id_conversacion", "orden_negocio", "op_text", "op_text_final", "nodo_final"]].head(10)

df_negocio (solo pasos de negocio, aristas recalculadas): (503607, 28)
id_conversacion con al menos un paso de negocio: 95525


,id_conversacion,orden_negocio,op_text,op_text_final,nodo_final
0,0000b8f3-39e8-4388-aa08-23c0031a9d9f,0,Inicio IVR,Habeas data negativo,Inicio IVR
1,0000b8f3-39e8-4388-aa08-23c0031a9d9f,1,Habeas data negativo,Agendar servicio de instalación,Habeas data negativo
2,0000b8f3-39e8-4388-aa08-23c0031a9d9f,2,Agendar servicio de instalación,Menu principal,Agendar servicio de instalación
3,0000b8f3-39e8-4388-aa08-23c0031a9d9f,3,Menu principal,Paso agente instalaciones,Menú Principal
4,0000b8f3-39e8-4388-aa08-23c0031a9d9f,4,Paso agente instalaciones,final,Paso agente instalaciones
5,0001de09-fc90-45b0-a909-29c583544af1,0,Inicio IVR,final,Inicio IVR
6,0001f83b-86fd-464d-a881-df45bb9181bf,0,Inicio IVR,Estado de entrega,Inicio IVR
7,0001f83b-86fd-464d-a881-df45bb9181bf,1,Estado de entrega,Menu principal,Estado de entrega
8,0001f83b-86fd-464d-a881-df45bb9181bf,2,Menu principal,Paso agente despachos,Menú Principal
9,0001f83b-86fd-464d-a881-df45bb9181bf,3,Paso agente despachos,¿Factura encontrada? = SI,Paso agente despachos


## Resumen por conversación: `traza_final_negocio` y `n_pasos_negocio`

In [ ]:
n_pasos_negocio = df_negocio.groupby("id_conversacion").size().rename("n_pasos_negocio")

traza_final_negocio = (
    df_negocio.groupby("id_conversacion")
    .agg(traza_final_negocio=("op_text", "last"))
)

df_resumen = pd.concat([n_pasos_negocio, traza_final_negocio], axis=1).reset_index()
print("Conversaciones resumidas:", len(df_resumen))
print("\nDistribución de n_pasos_negocio:")
print(df_resumen["n_pasos_negocio"].describe())
df_resumen.head(5)

Conversaciones resumidas: 95525

Distribución de n_pasos_negocio:
count    95525.000000
mean         5.271992
std          4.127812
min          1.000000
25%          3.000000
50%          4.000000
75%          6.000000
max         40.000000
Name: n_pasos_negocio, dtype: float64


,id_conversacion,n_pasos_negocio,traza_final_negocio
0,0000b8f3-39e8-4388-aa08-23c0031a9d9f,5,Paso agente instalaciones
1,0001de09-fc90-45b0-a909-29c583544af1,1,Inicio IVR
2,0001f83b-86fd-464d-a881-df45bb9181bf,6,¿Total de facturas mayor a uno? = SI
3,00020b12-08d0-4452-9b59-6a2116611636,1,Inicio IVR
4,0002144b-cde9-48eb-b887-1475b7b82c9c,3,Paso agente cambio dir fecha


## Cruce contra `Ultima traza` (auditoría — no bloquea el resto del notebook)

In [ ]:
ultima_traza = pd.read_excel(MAESTRO_PATH, sheet_name="Ultima traza")
ultima_traza["NombreUltimaTraza"] = ultima_traza["NombreUltimaTraza"].astype(str).str.strip()


def normalizar(texto):
    return unidecode(str(texto)).strip().lower()


ultima_traza["texto_normalizado"] = ultima_traza["NombreUltimaTraza"].apply(normalizar)
lookup_ultima_traza = ultima_traza.drop_duplicates(subset="texto_normalizado", keep="first").set_index("texto_normalizado")

df_resumen["texto_normalizado"] = df_resumen["traza_final_negocio"].apply(normalizar)
df_resumen = df_resumen.merge(
    lookup_ultima_traza[["Pasa a Agente", "Efectivo Ultima Traza", "ClasificaciónEfectivos"]],
    left_on="texto_normalizado",
    right_index=True,
    how="left",
)

print(f"% de conversaciones con traza_final_negocio identificado en 'Ultima traza': {df_resumen['Pasa a Agente'].notna().mean():.1%}")
print("\ntraza_final_negocio SIN identificar en 'Ultima traza' (top 20 por frecuencia):")
print(
    df_resumen[df_resumen["Pasa a Agente"].isna()]["traza_final_negocio"]
    .value_counts()
    .head(20)
)

% de conversaciones con traza_final_negocio identificado en 'Ultima traza': 58.3%

traza_final_negocio SIN identificar en 'Ultima traza' (top 20 por frecuencia):
traza_final_negocio
Inicio IVR inbound transfers                                            12980
Igual_O_Menor_30_Dias                                                    5508
Iniciar_Tu_Garantia                                                      4781
Estado_De_Tu_Garantia                                                    4540
Confirmar documento de la factura                                        3060
¿Total de facturas mayor a uno? = SI                                     2246
¿Fecha de contacto igual a fecha pactada? = NO                           1729
Paso agente cambio dir fecha                                             1620
Transportador Alkosto                                                    1047
¿Factura corresponde a ENDI? = NO                                         930
¿Estado igual a despachado o en tránsi

## Definición de escenarios

Mismos 5 escenarios de `01_Alk_final.ipynb`, con el mismo umbral de pasos (ahora
medido solo sobre pasos de negocio). Cada escenario acepta una **lista** de
valores de `traza_final_negocio` porque el crosswalk del Notebook 3 puede dejar
el texto interno viejo (`Igual_O_Menor_30_Dias`) conviviendo con el nombre del
maestro (`Periodo igual o menor a 30 dias`) — hay que capturar ambos.

In [ ]:
ESCENARIOS = {
    "menu": {
        "valores_traza_final": ["Menu principal"],
        "max_pasos": 4,
    },
    "transferencia": {
        "valores_traza_final": ["Transferencia Alkosto - Tuya"],
        "max_pasos": 4,
    },
    "producto": {
        "valores_traza_final": ["Existencia de producto"],
        "max_pasos": 5,
    },
    "entrega": {
        "valores_traza_final": ["Estado de entrega"],
        "max_pasos": 5,
    },
    "garantia_menor": {
        "valores_traza_final": ["Periodo igual o menor a 30 dias", "Igual_O_Menor_30_Dias"],
        "max_pasos": 6,
    },
}

for nombre, cfg in ESCENARIOS.items():
    n = df_resumen["traza_final_negocio"].isin(cfg["valores_traza_final"]).sum()
    print(f"{nombre}: {n} conversaciones con traza_final_negocio en {cfg['valores_traza_final']}")

menu: 19448 conversaciones con traza_final_negocio en ['Menu principal']
transferencia: 0 conversaciones con traza_final_negocio en ['Transferencia Alkosto - Tuya']
producto: 0 conversaciones con traza_final_negocio en ['Existencia de producto']
entrega: 0 conversaciones con traza_final_negocio en ['Estado de entrega']
garantia_menor: 5508 conversaciones con traza_final_negocio en ['Periodo igual o menor a 30 dias', 'Igual_O_Menor_30_Dias']


## Exportar `base_<escenario>_hasta_N.xlsx` / `_mas_N.xlsx`

In [ ]:
def exportar_escenario(nombre, valores_traza_final, max_pasos):
    ids_escenario = df_resumen.loc[
        df_resumen["traza_final_negocio"].isin(valores_traza_final), "id_conversacion"
    ]
    resumen_escenario = df_resumen[df_resumen["id_conversacion"].isin(ids_escenario)]

    ids_hasta = resumen_escenario.loc[resumen_escenario["n_pasos_negocio"] <= max_pasos, "id_conversacion"]
    ids_mas = resumen_escenario.loc[resumen_escenario["n_pasos_negocio"] > max_pasos, "id_conversacion"]

    base_hasta = df_negocio[df_negocio["id_conversacion"].isin(ids_hasta)]
    base_mas = df_negocio[df_negocio["id_conversacion"].isin(ids_mas)]

    path_hasta = ESCEN_DIR / f"base_{nombre}_hasta_{max_pasos}.xlsx"
    path_mas = ESCEN_DIR / f"base_{nombre}_mas_{max_pasos}.xlsx"
    base_hasta.to_excel(path_hasta, index=False)
    base_mas.to_excel(path_mas, index=False)

    print(
        f"{nombre}: {len(ids_hasta)} conversaciones <= {max_pasos} pasos -> {path_hasta.name} | "
        f"{len(ids_mas)} conversaciones > {max_pasos} pasos -> {path_mas.name}"
    )
    return path_hasta, path_mas


rutas_generadas = {}
for nombre, cfg in ESCENARIOS.items():
    rutas_generadas[nombre] = exportar_escenario(nombre, cfg["valores_traza_final"], cfg["max_pasos"])

menu: 17656 conversaciones <= 4 pasos -> base_menu_hasta_4.xlsx | 1792 conversaciones > 4 pasos -> base_menu_mas_4.xlsx
transferencia: 0 conversaciones <= 4 pasos -> base_transferencia_hasta_4.xlsx | 0 conversaciones > 4 pasos -> base_transferencia_mas_4.xlsx
producto: 0 conversaciones <= 5 pasos -> base_producto_hasta_5.xlsx | 0 conversaciones > 5 pasos -> base_producto_mas_5.xlsx
entrega: 0 conversaciones <= 5 pasos -> base_entrega_hasta_5.xlsx | 0 conversaciones > 5 pasos -> base_entrega_mas_5.xlsx
garantia_menor: 3136 conversaciones <= 6 pasos -> base_garantia_menor_hasta_6.xlsx | 2372 conversaciones > 6 pasos -> base_garantia_menor_mas_6.xlsx


## Extra: catch-all de "Estado de entrega" en cualquier punto de la conversación

Equivalente al filtro `opcionesnavegaciontrazaopciones.str.contains('Estado de
entrega')` de `01_Alk_final.ipynb` — captura conversaciones donde el cliente pasó
por `Estado de entrega` en algún momento, no solo si terminó ahí. Útil para el
grafo de ese sub-árbol completo (`base_grafos_estado_de_entrega.xlsx` en el
original).

In [ ]:
ids_con_estado_entrega = df_negocio.loc[
    df_negocio["op_text"].str.contains("Estado de entrega", case=False, na=False), "id_conversacion"
].unique()

base_estado_entrega = df_negocio[df_negocio["id_conversacion"].isin(ids_con_estado_entrega)]
path_estado_entrega = ESCEN_DIR / "base_grafos_estado_de_entrega.xlsx"
base_estado_entrega.to_excel(path_estado_entrega, index=False)

print(f"{len(ids_con_estado_entrega)} conversaciones pasaron por 'Estado de entrega' en algún punto -> {path_estado_entrega.name}")

16805 conversaciones pasaron por 'Estado de entrega' en algún punto -> base_grafos_estado_de_entrega.xlsx


---
**Antes de seguir al Notebook 5:**
1. Revisa que los conteos de cada escenario tengan sentido (ningún escenario en 0
   conversaciones — si alguno da 0, el nombre en `valores_traza_final` puede haber
   cambiado en el flujo actual y hay que ajustarlo).
2. Revisa el % de `traza_final_negocio` identificado contra `Ultima traza` — si es
   bajo, puede haber más renombres que agregar al crosswalk del Notebook 3.

**Siguiente paso:** `05_grafos.ipynb` — igual que `02_Grafos.ipynb` original, solo
cambiando la ruta de lectura por cada `base_*.xlsx` generado acá.